<a href="https://colab.research.google.com/github/ChiaHeJun/hy/blob/main/WalkthroughTask2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walkthrough: Task 2 - Treasure Game

Object-oriented programming. Run this notebook **from the Suggested Solution folder**, top to bottom. Each section follows the same shape: what the question is really asking, the thinking that leads to the design, then the solution built up and run. The goal is not to memorise the code but to be able to re-derive it.

---
## Task 2.1 - model the board pieces

**What is asked.** A `Tile` class with a symbol, and `Treasure` and `Bomb` classes that extend it.

**Thinking in objects.** Ask what every board piece has in common, and what is special. Every piece shows a symbol on the grid, so `symbol` and `get_symbol()` belong in the base class `Tile`, written once. A treasure additionally has a name; a bomb additionally has a damage value. So `Treasure` and `Bomb` **inherit** from Tile and add only their own attribute and accessor.

The one line that trips people up is `super().__init__(sym)`: the subclass receives the symbol but the base class is the one that knows how to store it, so pass it up. Skip that line and `get_symbol()` fails on every Treasure and Bomb, and with it most of the grid display.

In [ ]:
# Task 2.1
class Tile:
    def __init__(self,sym):
        self.symbol = sym

    def get_symbol(self):
        return self.symbol

class Bomb(Tile):
    def __init__(self, sym, dam):
        super().__init__(sym)
        self.damage = dam

    def get_damage(self):
        return self.damage

class Treasure(Tile):
    def __init__(self, sym, trea):
        super().__init__(sym)
        self.treasure = trea

    def get_treasure(self):
        return self.treasure

In [ ]:
# Quick check that the model behaves before building anything on it
t = Treasure('T', "Silver key")
b = Bomb('B', 1)
print(t.get_symbol(), t.get_treasure())
print(b.get_symbol(), b.get_damage())
print(isinstance(t, Tile), isinstance(b, Tile))

Both objects answer `get_symbol()` through inheritance and their own accessor through their subclass. Testing the model **before** the game loop uses it means any later bug is in the game loop, not the classes.

---
## Task 2.2 - display the grid

**What is asked.** Print the 5 by 5 grid, symbols separated by spaces, one row per line.

**Thinking.** The grid is a list of rows; a row is a list of Tile objects. Two facts follow: displaying needs **two** loops (rows, then tiles within a row), and what gets printed is `tile.get_symbol()`, never the tile itself. `print(row)` or `print(*grid)` shows Python object noise, not the board. `end=' '` keeps a row on one line; the bare `print()` after the inner loop starts the next row.

In [ ]:
# Task 2.2
def display_grid(grid):
    for row in grid:
        for tile in row:
            print(tile.get_symbol(), end=' ')
        print()

---
## Task 2.3 - the game

**What is asked.** Build the board, place three named treasures, a bomb and the player, then process a whole sequence of W/A/S/D moves and announce the outcome.

This is the largest function in the paper, so the design matters more than the typing. Break it into five phases and get each right on its own.

**Phase 1 - the board.** 25 cells, each its **own** `Tile('.')` object. Then place the specials **by assignment**: `grid[1][2] = Treasure('T', "Silver key")` replaces the plain tile at row 1, column 2. Reading `grid[1][2]` into a variable does nothing to the board; several candidates lost the placement marks exactly there. Note the coordinate discipline: first index row, second column, everywhere, with W/S changing the row and A/D the column.

**Phase 2 - state.** What does the game need to remember? Player position (`p_row`, `p_col`), what has been collected (`inventory`, a list), and `lives = 1`.

**Phase 3 - input.** The whole sequence arrives in **one** `input()` call, and a string is already iterable, so `for move in moves:` handles it character by character.

**Phase 4 - one move.** The elegant trick in the suggested solution: compute the move into **temporary** variables first, and only commit them after the boundary check passes. Try-then-commit means an out-of-bounds move simply leaves the player where they stood, with no undo logic. Without any check, `p_row` can reach -1, and Python silently wraps `grid[-1]` to the bottom row, a bug that produces no error message at all. After committing, read the symbol at the new cell: `'B'` costs `get_damage()` lives (the object says how much; hardcoding `1` ignores the object model), `'T'` adds `get_treasure()` to the inventory, guarded by `not in` so a revisited cell does not count twice.

**Phase 5 - the verdict.** After the loop: no lives is Game Over; three treasures is a win; anything else did not win. The wording is checked word for word, and the treasure count and names come from the inventory itself: `str(len(inventory))` and `", ".join(inventory)`.

In [ ]:
# Task 2.3 - main program
def main():
    # Create grid
    grid = [[Tile('.') for i in range(5)] for j in range(5)]

    # Add special tiles
    grid[1][2] = Treasure('T',"Silver key")
    grid[2][4] = Treasure('T',"Gold coin")
    grid[3][3] = Treasure('T',"Jade pendant")
    grid[2][3] = Bomb('B',1)
    grid[0][0] = Tile('P')
    display_grid(grid)
    print()

    # player variables
    p_row, p_col = 0, 0
    inventory = []
    lives = 1

    moves = input("Enter sequence of moves (W/A/S/D):")

    for move in moves:
        p_row_temp = p_row   # temp row and col
        p_col_temp = p_col

        # move player position
        if move == 'W': # up
            p_row_temp -= 1
        elif move == 'A':  # left
            p_col_temp -= 1
        elif move == 'S':  # down
            p_row_temp += 1
        elif move == 'D':  # right
            p_col_temp += 1

        # check for boundary, treasure and bomb
        if 0<=p_row_temp<5 and 0<=p_col_temp<5:  # within boundary
            p_row = p_row_temp  # update row and col if within boundary
            p_col = p_col_temp

            symbol = grid[p_row][p_col].get_symbol()

            if symbol == 'B':  # bomb
                lives -= grid[p_row][p_col].get_damage()

            elif symbol == 'T':   # treasure
                treasure = grid[p_row][p_col].get_treasure()
                if treasure not in inventory:
                    inventory.append(treasure)

    if lives <= 0:  # No more lives
        print("Game Over, no more lives.")
    elif len(inventory)==3:  # Won
        print("Congratulations! You won the game!")
        print("Treasures collected " + str(len(inventory)) + "/3:" , ", ".join(inventory))
    else:  # Did not win
        print("Sorry, you did not win the game.")
        print("Treasures collected " + str(len(inventory)) + "/3:", ", ".join(inventory))

**Reading the three sample runs as test cases.** They are not arbitrary: `SDDDDSSA` collects all three treasures (win), `WASDDDDSSSA` starts with moves that push off the board (exercising the boundary check) and ends short of three treasures, and `SDDDSSDW` walks into the bomb (game over). In the exam you call `main()` three times and type each sequence; the cells below script the same inputs so the notebook runs end to end.

In [ ]:
from unittest import mock

print("Sample run 1 (Won the game), simulating input SDDDDSSA:")
with mock.patch("builtins.input", side_effect=["SDDDDSSA"]):
    main()

In [ ]:
print("Sample run 2 (Did not win, includes out-of-boundary moves), simulating input WASDDDDSSSA:")
with mock.patch("builtins.input", side_effect=["WASDDDDSSSA"]):
    main()

In [ ]:
print("Sample run 3 (Game Over), simulating input SDDDSSDW:")
with mock.patch("builtins.input", side_effect=["SDDDSSDW"]):
    main()

---
## The thread through Task 2

The classes exist so the game loop can ask objects for their data: `get_damage()` instead of a hardcoded 1, `get_treasure()` instead of a remembered name. When the loop honours the object model, each of the fourteen marking points in 2.3 falls out of one small, testable decision: try-then-commit movement, symbol first then accessor, and a guarded append.